# QUERY 3
Cuenta de origen y monto de transacciones USD en el período [2022-09-06, 2022-09-15]
con monto menor a 1 centésimo del promedio encontrado para el mismo formato de
pago en el período [2022-09-01, 2022-09-05]

In [ ]:
import pandas as pd

RUTA_DE_DATASETS             = "../../datasets"
NOMBRE_DATASET      = "transacciones_sample.csv"
RUTA_RESULTADO_QUERY3        = "q3_solucion.csv"

transacciones_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_DATASET}",
    dtype={"From Bank": "string", "Account": "string",
           "To Bank": "string", "Account.1": "string"}
)

guardar_csv = lambda df, ruta: df.to_csv(ruta, index=False)
# Filtrar USD antes de samplear
transacciones = transacciones_completas[
    transacciones_completas["Payment Currency"] == "US Dollar"
]

Sample generada: 120 filas USD → q3_sample.csv


In [ ]:
transacciones["Timestamp"]   = pd.to_datetime(transacciones["Timestamp"])
transacciones["Amount Paid"] = pd.to_numeric(transacciones["Amount Paid"], errors="coerce")
transacciones = transacciones.dropna(subset=["Amount Paid", "Payment Format"])

periodo_temprano = transacciones[
    (transacciones["Timestamp"] >= "2022-09-01") &
    (transacciones["Timestamp"] < "2022-09-06")
]
periodo_tardio = transacciones[
    (transacciones["Timestamp"] >= "2022-09-06") &
    (transacciones["Timestamp"] < "2022-09-16")
]

print(f"Período temprano (1/9-5/9): {len(periodo_temprano)} transacciones")
print(f"Período tardío  (6/9-15/9): {len(periodo_tardio)} transacciones")

Período temprano (1/9-5/9): 55 transacciones
Período tardío  (6/9-15/9): 58 transacciones


In [42]:
import hashlib

def obtener_id_shard(valor, total_shards):
    valor_norm = str(valor).strip()
    hash_hex = hashlib.md5(valor_norm.encode('utf-8')).hexdigest()
    return (int(hash_hex, 16) % total_shards) + 1

formatos = ["ACH", "Wire", "Cheque", "Cash", "Credit Card", "Reinvestment"]
for f in formatos:
    print(f"{f} -> shard {obtener_id_shard(f, 3)}")

ACH -> shard 3
Wire -> shard 2
Cheque -> shard 3
Cash -> shard 3
Credit Card -> shard 2
Reinvestment -> shard 3


In [43]:
stats_por_formato = (
    periodo_temprano
    .groupby("Payment Format")["Amount Paid"]
    .agg(suma="sum", count="count")
)
stats_por_formato["promedio"] = stats_por_formato["suma"] / stats_por_formato["count"]

print(stats_por_formato)

                     suma  count      promedio
Payment Format                                
ACH              50608.86      8   6326.107500
Cash             27228.11      7   3889.730000
Cheque          235454.77      9  26161.641111
Credit Card      34855.76     17   2050.338824
Reinvestment    585712.96     10  58571.296000
Wire             43231.53      4  10807.882500


In [44]:
df = periodo_tardio.copy()
df["Promedio Formato"] = df["Payment Format"].map(stats_por_formato["promedio"])
df = df.dropna(subset=["Promedio Formato"])

resultado_query3 = df[
    df["Amount Paid"] < df["Promedio Formato"] * 0.01
][["Account", "Amount Paid"]].rename(columns={"Account": "From Account"}).reset_index(drop=True)

guardar_csv(resultado_query3, RUTA_RESULTADO_QUERY3)
print(f"Resultados Q3: {len(resultado_query3)} transacciones")
resultado_query3.head()

Resultados Q3: 11 transacciones


,From Account,Amount Paid
0,8001DC570,0.82
1,803019BB0,4.16
2,80D052E10,14.66
3,10042B660,106.17
4,80A9E9DD0,9.71


In [45]:
# Validación del resultado
df_validacion = resultado_query3.copy()
df_validacion = df_validacion.merge(
    periodo_tardio[["Account", "Payment Format", "Timestamp", "Amount Paid"]].rename(columns={"Account": "From Account"}),
    on=["From Account", "Amount Paid"],
    how="left"
)
df_validacion["Promedio Formato"] = df_validacion["Payment Format"].map(stats_por_formato["promedio"])
df_validacion["1% Promedio"] = df_validacion["Promedio Formato"] * 0.01
df_validacion["Es valido"] = df_validacion["Amount Paid"] < df_validacion["1% Promedio"]

print(f"Filas totales: {len(df_validacion)}")
print(f"Filas válidas: {df_validacion['Es valido'].sum()}")
print(f"Filas inválidas: {(~df_validacion['Es valido']).sum()}")
df_validacion.head(10)

Filas totales: 11
Filas válidas: 11
Filas inválidas: 0


,From Account,Amount Paid,Payment Format,Timestamp,Promedio Formato,1% Promedio,Es valido
0,8001DC570,0.82,Wire,2022-09-10 05:27:00,10807.882500,108.078825,True
1,803019BB0,4.16,ACH,2022-09-06 03:34:00,6326.107500,63.261075,True
2,80D052E10,14.66,Credit Card,2022-09-06 10:51:00,2050.338824,20.503388,True
3,10042B660,106.17,Cheque,2022-09-07 10:38:00,26161.641111,261.616411,True
4,80A9E9DD0,9.71,Credit Card,2022-09-07 00:13:00,2050.338824,20.503388,True
5,80CEF7210,26.80,Wire,2022-09-09 04:39:00,10807.882500,108.078825,True
6,8116BAC00,7.73,ACH,2022-09-08 06:52:00,6326.107500,63.261075,True
7,80EE8FCD0,212.89,Cheque,2022-09-06 21:20:00,26161.641111,261.616411,True
8,80230B070,153.71,Cheque,2022-09-06 13:13:00,26161.641111,261.616411,True
9,817240790,5.07,Credit Card,2022-09-09 00:26:00,2050.338824,20.503388,True


In [46]:
print(f"Período temprano total: {len(periodo_temprano)}")
print(f"Período tardío total: {len(periodo_tardio)}")
print("\nTemprano por formato:")
print(periodo_temprano.groupby("Payment Format")["Amount Paid"].count())
print("\nTardío por formato:")
print(periodo_tardio.groupby("Payment Format")["Amount Paid"].count())

Período temprano total: 55
Período tardío total: 58

Temprano por formato:
Payment Format
ACH              8
Cash             7
Cheque           9
Credit Card     17
Reinvestment    10
Wire             4
Name: Amount Paid, dtype: int64

Tardío por formato:
Payment Format
ACH            10
Cash            1
Cheque         26
Credit Card    19
Wire            2
Name: Amount Paid, dtype: int64
